# Comprobación de sanidad: motor *paper trading* vs. predicciones offline congeladas

Gate obligatorio **antes** del *paper trading* live. Consume el `promotion_manifest.json` generado por `01_tabla_promocion.ipynb` y, para cada celda campeona:

1. Carga el último fold externo del estudio promovido.
2. Reconstruye la ventana de *features* y aplica el mismo `FrozenProductionModel` que usa `FrozenSignalEngine` en runtime.
3. Compara la señal direccional (`pred_label_side`) contra `predictions_test.parquet` fila a fila.

**Criterio de aprobación**

- `lstm` / `xgboost`: tasa de acuerdo >= 90 % (configurable).
- `residual_hybrid`: tasa >= 75 % por tolerancia documentada del refit de `lstm_stage1_state.pt`.

Si alguna celda falla, **no** se autoriza el arranque del *paper trading*. El CLI homólogo es `scripts/run_live_signal_sanity_check.py`. Tras superar el gate, el siguiente paso operativo es `03_postmortem_paper_trading.ipynb` (postmortem live) o la retrospectiva en `04_retrospectiva_post_freeze.ipynb`.

**Productos**

- `reports/validation/promotion/20260601T131904Z/signal_sanity_check.json` — informe JSON con umbrales, resultados por símbolo y flag `all_pass` (misma carpeta que el manifiesto de promoción).

In [13]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'scripts' / 'run_live_signal_sanity_check.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'scripts' / 'run_live_signal_sanity_check.py').exists():
    raise SystemExit('No se encuentra el repo TFG.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib.util

_sanity_path = REPO_ROOT / 'scripts' / 'run_live_signal_sanity_check.py'
_spec = importlib.util.spec_from_file_location('run_live_signal_sanity_check', _sanity_path)
_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_mod)

DEFAULT_MIN_MATCH_RATE = _mod.DEFAULT_MIN_MATCH_RATE
RESIDUAL_HYBRID_MIN_MATCH_RATE = _mod.RESIDUAL_HYBRID_MIN_MATCH_RATE
_check_one_cell = _mod._check_one_cell

from src.live.promotion_loader import load_promotion_manifest

COLUMNAS_ES = {
    'symbol': 'simbolo',
    'lane': 'carril',
    'engine_family': 'familia_motor',
    'fold': 'fold',
    'match_rate': 'tasa_acuerdo',
    'n_rows': 'n_filas',
    'passes': 'pasa_gate',
    'error': 'error',
    'matches': 'coincidencias',
    'min_match_rate_used': 'umbral_acuerdo_usado',
    'max_abs_proba_diff': 'max_diff_prob_abs',
}

COLUMNAS_RESUMEN_ES = {
    'n': 'n_simbolos',
    'min_match': 'min_tasa_acuerdo',
    'max_abs_proba_diff': 'max_diff_prob_abs',
}


def _a_es(df: pd.DataFrame, columnas: list[str] | None = None) -> pd.DataFrame:
    """Renombra columnas visibles al español sin alterar el dict de resultados."""
    out = df.copy()
    if columnas is not None:
        out = out[[c for c in columnas if c in out.columns]]
    renombrar = {k: v for k, v in COLUMNAS_ES.items() if k in out.columns}
    return out.rename(columns=renombrar)

PROMOTION_MANIFEST = REPO_ROOT / 'reports' / 'validation' / 'promotion' / '20260601T131904Z' / 'promotion_manifest.json'
SANITY_JSON = PROMOTION_MANIFEST.parent / 'signal_sanity_check.json'
PROMOTION_MANIFEST.exists(), SANITY_JSON.exists()

(True, True)

## 1. Ejecutar el gate sobre el manifiesto campeón

In [10]:
cells = load_promotion_manifest(PROMOTION_MANIFEST, repo_root=REPO_ROOT)

results = []
for cell in cells.values():
    res = _check_one_cell(
        cell,
        min_match_rate=DEFAULT_MIN_MATCH_RATE,
    )
    results.append(res)

df = pd.DataFrame(results)
_a_es(
    df,
    ['symbol', 'lane', 'engine_family', 'fold', 'match_rate', 'n_rows', 'passes', 'error'],
)

,simbolo,carril,familia_motor,fold,tasa_acuerdo,n_filas,pasa_gate,error
0,BNBUSDT,primary,residual_hybrid,fold_094,1.000000,139,True,None
1,BTCUSDT,primary,lstm,fold_096,1.000000,139,True,None
2,ETHUSDT,primary,residual_hybrid,fold_096,0.848921,139,True,None
3,SOLUSDT,primary,residual_hybrid,fold_060,1.000000,139,True,None
4,XRPUSDT,metalabeling,lstm,fold_088,1.000000,133,True,None


## 2. Resumen del gate y persistencia del informe

In [11]:
all_pass = bool(df['passes'].all())
report = {
    'promotion_manifest': PROMOTION_MANIFEST.as_posix(),
    'min_match_rate': DEFAULT_MIN_MATCH_RATE,
    'residual_hybrid_min_match_rate': RESIDUAL_HYBRID_MIN_MATCH_RATE,
    'all_pass': all_pass,
    'results': results,
}
SANITY_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

print('Gate superado (todas las celdas):' if all_pass else 'Gate fallido — no lanzar paper trading', all_pass)
print('Informe:', SANITY_JSON.relative_to(REPO_ROOT))
(
    df.groupby('engine_family')
    .agg(
        n=('symbol', 'count'),
        min_match=('match_rate', 'min'),
        max_abs_proba_diff=('max_abs_proba_diff', 'max'),
    )
    .rename(columns=COLUMNAS_RESUMEN_ES)
    .rename_axis('familia_motor')
)

Gate superado (todas las celdas): True
Informe: reports/validation/promotion/20260601T131904Z/signal_sanity_check.json


,n_simbolos,min_tasa_acuerdo,max_diff_prob_abs
familia_motor,,,
lstm,2,1.000000,1.200253e-07
residual_hybrid,3,0.848921,4.737398e-08


## 3. Evidencia por símbolo (probabilidades offline vs live)

La comparación contractual es por **lado** (`pred_label_side`), no por probabilidad exacta.

In [12]:
detail_cols = [
    'symbol', 'lane', 'engine_family', 'fold', 'matches', 'n_rows',
    'match_rate', 'min_match_rate_used', 'max_abs_proba_diff', 'passes',
]
_a_es(df, detail_cols).sort_values('tasa_acuerdo')

,simbolo,carril,familia_motor,fold,coincidencias,n_filas,tasa_acuerdo,umbral_acuerdo_usado,max_diff_prob_abs,pasa_gate
2,ETHUSDT,primary,residual_hybrid,fold_096,118,139,0.848921,0.75,4.737398e-08,True
0,BNBUSDT,primary,residual_hybrid,fold_094,139,139,1.000000,0.75,2.028361e-08,True
1,BTCUSDT,primary,lstm,fold_096,139,139,1.000000,0.90,5.305084e-08,True
3,SOLUSDT,primary,residual_hybrid,fold_060,139,139,1.000000,0.75,4.205732e-08,True
4,XRPUSDT,metalabeling,lstm,fold_088,133,133,1.000000,0.90,1.200253e-07,True


## 4. Conclusiones

- Las **5** celdas campeonas del manifiesto `20260601T131904Z` (salida de `01_tabla_promocion.ipynb`) superan el gate (`all_pass=True`).
- **ETHUSDT** (`residual_hybrid`) registra la tasa mínima (**84.89 %**), por encima del umbral relajado de 75 %; el resto alcanza **100 %** de acuerdo direccional.
- Las diferencias de probabilidad (`max_abs_proba_diff`) permanecen del orden de `1e-7`; la comparación contractual es por **lado**, no por probabilidad exacta.
- El informe queda persistido en `reports/validation/promotion/20260601T131904Z/signal_sanity_check.json`, listo como evidencia previa al live.